# Honest Full-Precision Eval — Path A (Unsloth match training stack)

**Цель**: устранить inference-artifact в сравнении base vs GSPO vs KTO. Все три модели прогоняются через **Unsloth FastLanguageModel** (как в training notebook) с identical decoding protocol. Phase 0a: programmatic correctness only (fast_mode=True); Phase 0b — Cerebras judge поверх saved completions, отдельным async скриптом.

**Compute**: Colab A100 40GB. Estimated:
- С `causal-conv1d`: ~30 tok/s × 2048 max × 143 problems × 3 stages ≈ **5h**.
- Без `conv1d` (fla one): ~12 tok/s × 2048 × 143 × 3 ≈ **12h** или 50 stratified subset за **4h**.

**Output**: `evaluation/reports/honest_full_precision_phase0a_YYYYMMDD.json` (repo + Drive mirror).

**Стек (verified 2026-05-05)**: torch 2.10.0+cu128, transformers 5.5.0, trl 0.24.0, datasets 4.3.0, unsloth 2026.5.1, fla 0.5.0+. ALL strict-pinned под Unsloth constraints.

**Структура**:
1. **Setup** — install verified stack → **RESTART RUNTIME** → re-run cell to load imports
2. **DECODING_CONFIG** — single-protocol для трёх моделей (num_predict=2048, enable_thinking=True, temperature=0.0)
3. **Load eval dataset** — 143 calc problems (numeric + latex_boxed)
4. **Per-model inference** — base / +GSPO / +KTO via FastLanguageModel + PEFT merge
5. **Eval loop** — fast_mode=True (programmatic correctness, no Cerebras), per-problem timing для первых 3 problems
6. **Run + save** — JSON report со всеми completions для Phase 0b later


In [ ]:
# Cell 1: Setup (FINAL — empirically verified Unsloth-compatible stack)
# First run: install pinned versions → RESTART RUNTIME → re-run this cell.
# Sentinel /content/.install_done_v4 gates install; after restart imports load.
import subprocess, os

# ─── GPU check (need ≥24GB for 9B bf16) ────────────────────
gpu_info = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']
).decode().strip()
print(f'GPU: {gpu_info}')
gpu_memory_mib = int(gpu_info.split(',')[1].strip().split()[0])
assert gpu_memory_mib >= 24_000, (
    f'Insufficient VRAM for 9B bf16: {gpu_memory_mib} MiB. Need >=24GB.'
)
print(f'VRAM check passed: {gpu_memory_mib} MiB ({gpu_memory_mib/1024:.1f} GiB)')

# ─── Drive mount + repo clone (idempotent) ────────────────
from google.colab import drive
drive.mount('/content/drive')
if not os.path.exists('/content/MITS'):
    !git clone https://github.com/Siesher/MIST.git /content/MITS
%cd /content/MITS
!git checkout 019-ns-vstar-dpo && git pull origin 019-ns-vstar-dpo

# ─── Install (FIRST RUN ONLY — gated by sentinel v4) ──────
# Empirically verified stack — все версии strict-pinned под Unsloth 2026.5.1
# constraint set: transformers<=5.5.0, torch<2.11.0, trl<=0.24.0, datasets<4.4.0
SENTINEL = '/content/.install_done_v4'
if not os.path.exists(SENTINEL):
    print('=== Installing verified stack (Unsloth + transformers 5.5.0 + torch 2.10+cu128) ===')

    # Step 1: pytorch ecosystem на torch 2.10+cu128 (Unsloth max + matches system nvcc 12.8)
    # --extra-index-url нужен для cu128 wheels (на PyPI mirror)
    print('Step 1: pinning torch 2.10+cu128...')
    !pip install -q --force-reinstall \
        "torch==2.10.0" "torchvision" "torchaudio" \
        --extra-index-url https://download.pytorch.org/whl/cu128

    # Step 2: HF stack — strict pin к Unsloth-compatible versions
    # transformers 5.5.0 — последняя в Unsloth bound, имеет qwen3_5 architecture
    print('Step 2: HF stack (transformers 5.5.0, trl 0.24, datasets 4.3)...')
    !pip install -q --force-reinstall \
        "transformers==5.5.0" "trl==0.24.0" "datasets==4.3.0" \
        "huggingface_hub" "tokenizers"
    !pip install -q peft accelerate bitsandbytes

    # Step 3: Unsloth core (2026.5.1 — latest, supports qwen3_5)
    print('Step 3: Unsloth core...')
    !pip install -q --upgrade unsloth unsloth_zoo

    # Step 4: Other deps
    print('Step 4: scipy + utilities...')
    !pip install -q --upgrade scipy
    !pip install -q sentencepiece protobuf loguru python-dotenv openai
    !pip install -q sympy chempy

    # Step 5: flash-linear-attention — Triton-based GDN kernels (always works, no CUDA build)
    # Покрывает 24/32 GDN linear-attention layers в Qwen3.5-9B
    print('Step 5: flash-linear-attention (Triton)...')
    !pip install -q flash-linear-attention 2>&1 | tail -3

    # Step 6: causal-conv1d — best-effort. Build часто падает на Colab без точного
    # nvcc/torch ABI match, но fla одной достаточно для recurrent rule (главная часть GDN).
    # Без conv1d ожидаем ~10-15 tok/s; с conv1d ~25-35 tok/s.
    print('Step 6: causal-conv1d (best-effort — fallback OK если build fails)...')
    !pip install -q causal-conv1d 2>&1 | tail -3 || echo '  conv1d build failed — fla одной хватит для GDN'

    # Step 7: Kill torchcodec — sentence_transformers ловит только (ImportError, OSError),
    # но torchcodec runtime DLL fail → RuntimeError → cascades в unsloth import.
    # Uninstall переводит fail в ImportError → caught → AudioDecoder=None → load OK.
    print('Step 7: remove torchcodec (sentence_transformers fallback нужен ImportError, не RuntimeError)...')
    !pip uninstall -y torchcodec 2>&1 | tail -1

    # Sentinel: stdlib open() — НЕ требует Path import.
    open(SENTINEL, 'w').close()
    print('=' * 60)
    print('  Install complete. RESTART RUNTIME NOW:')
    print('    Runtime → Restart session')
    print('  After restart: re-run THIS cell — install will skip, imports will load.')
    print('=' * 60)
    raise SystemExit('Restart required — re-run this cell after Runtime → Restart session.')

# ─── Post-restart imports ─────────────────────────────────
# CRITICAL: import unsloth BEFORE transformers — Unsloth patches transformers internals
# at import time. Reverse order gives "Unsloth should be imported before transformers"
# warning + missed optimizations.
import unsloth  # noqa: F401 — must precede transformers import

import sys, json, time, logging
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List

import torch
import transformers
import huggingface_hub
from unsloth import FastLanguageModel
from peft import PeftModel
from dotenv import load_dotenv

# Sanity: pinned versions должны match — иначе runtime ABI mismatch.
print(f'transformers: {transformers.__version__} | huggingface_hub: {huggingface_hub.__version__} | torch: {torch.__version__}')
assert transformers.__version__.startswith('5.5'), (
    f'Expected transformers 5.5.x (Unsloth max + qwen3_5 support). '
    f'Got {transformers.__version__}. If just installed — Runtime → Restart session.'
)
assert torch.__version__.startswith('2.10'), (
    f'Expected torch 2.10.x (Unsloth requires <2.11). Got {torch.__version__}. '
    f'If just downgraded — Runtime → Restart session.'
)

PROJECT_ROOT = Path('/content/MITS')
sys.path.insert(0, str(PROJECT_ROOT))

# Load .env from Drive (Cerebras keys + optional HF_TOKEN для приватных adapter'ов)
ENV_PATH = Path('/content/drive/MyDrive/MITS_secrets/.env')
if ENV_PATH.exists():
    load_dotenv(ENV_PATH)
    print(f'Loaded env from {ENV_PATH}')
    if os.environ.get('HF_TOKEN'):
        from huggingface_hub import login as hf_login
        hf_login(token=os.environ['HF_TOKEN'])
        print('HF authenticated via .env')
    else:
        print('Warning: HF_TOKEN missing in .env — adapter download может 401 если приватные.')
else:
    print(f'No .env at {ENV_PATH}. Cerebras judge будет fail в Phase 0b.')

from training.scripts.evaluate_stage import (
    SYSTEM_PROMPT_CALC,
    extract_answer,
    check_format_compliance,
    evaluate_combined_quality,
    load_eval_dataset,
)
from training.cerebras_client import CerebrasClient

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('honest_eval')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
logger.info(f'Device: {DEVICE} | bf16: {torch.cuda.is_bf16_supported()}')


## Cell 3 — Decoding Configuration (зафиксирован)

Параметры применяются ОДИНАКОВО ко всем трём моделям (base, GSPO, KTO). Зафиксировано:

| Параметр | Значение | Обоснование |
|----------|----------|-------------|
| `num_predict` | 4096 | Покрывает 95-percentile thinking длин (GSPO учился с budget=2048; 4096 даёт запас на hard problems без overhead 8192). |
| `enable_thinking` | `True` | Матчит training distribution GSPO/KTO + native режим Qwen3.5-9B. False нивелировал бы RL-effect целиком — unfair. |
| `temperature` | 0.0 | Greedy для deterministic accuracy. Diversity sampling — Phase 1 (V-STaR). |
| `system_prompt` | `SYSTEM_PROMPT_CALC` | Apple-to-apple с предыдущими compare_base_vs_gspo отчётами. |

**Что это даёт для диплома**: section "Methodology — inference protocol" в одну таблицу. Альтернатива (запустить второй раз с `enable_thinking=False`) — рассматривается как _Appendix-grade ablation_, если останется compute после Phase 1-3.

In [ ]:
# Cell 4: Decoding config (filled — single-protocol run)
# Same config applied to all three models (base, GSPO, KTO).

DECODING_CONFIG = {
    # 2048: matches GSPO/KTO training budget exactly. Avoids truncation на
    # hard problems (training distribution). На steady-state 13 tok/s
    # 2048 / 13 = ~157s worst case per problem; avg ~80-120s due к early
    # termination on </think>. 143 × 100s × 3 stages = ~12h per night, run
    # split across 2 nights если нужно. Methodologically chistyy.
    'num_predict': 2048,
    # True: matches GSPO/KTO training distribution (chat_template_kwargs.enable_thinking
    # =True in grpo_qwen3_5_9b_(8).ipynb). Qwen3.5-9B base also natively supports <think>.
    # False would nullify the RL effect — unfair to GSPO/KTO. Keep one consistent mode.
    'enable_thinking': True,
    # 0.0: greedy decoding for accuracy eval. Diversity sampling is for V-STaR (Phase 1).
    'temperature': 0.0,
    # Same calc system prompt used in compare_base_vs_gspo_*.json — preserves apple-to-
    # apple with prior reports for the Cerebras-only path; only inference layer changes.
    'system_prompt': SYSTEM_PROMPT_CALC,
    'rationale': (
        'Single thinking-on protocol mirrors training distribution + production deployment. '
        '2048 budget covers GSPO training distribution. Greedy decoding for deterministic accuracy.'
    ),
}

assert all(v is not None for v in DECODING_CONFIG.values()), 'Fill in DECODING_CONFIG keys'
logger.info(f'Decoding config: {DECODING_CONFIG}')

In [ ]:
# Cell 5: Load eval dataset — calc subset only (numeric / latex_boxed)
EVAL_PATH = PROJECT_ROOT / 'training/data/eval_dataset.jsonl'
all_problems = load_eval_dataset(str(EVAL_PATH))
calc_problems = [p for p in all_problems if p.get('answer_type') in ('numeric', 'latex_boxed')]
logger.info(f'Total: {len(all_problems)} | Calc subset: {len(calc_problems)}')
# Sanity
from collections import Counter
logger.info(f'By domain: {Counter(p["domain"] for p in calc_problems)}')
logger.info(f'By difficulty: {Counter(p["difficulty"] for p in calc_problems)}')

In [ ]:
# Cell 5: Model loading via Unsloth (matches training stack from grpo_qwen3.5_9b.ipynb)
BASE_MODEL_ID = 'Qwen/Qwen3.5-9B'  # matches BASE_MODEL in training notebook
MAX_SEQ_LENGTH = 4096  # covers thinking budget 2048 + completion + headroom

ADAPTERS = {
    'base': None,
    'gspo': 'Siesher/mits-qwen3-9b-gspo',
    'kto':  'Siesher/mits-qwen3-9b-kto',
}


def load_model_with_adapter(adapter_id: str | None):
    """Load Qwen3.5-9B via Unsloth FastLanguageModel, optionally apply LoRA + merge.

    Why Unsloth: Qwen3.5-9B is image-text-to-text (multimodal) с Model Class
    AutoModelForImageTextToText. AutoModelForCausalLM не работает напрямую.
    FastLanguageModel.from_pretrained абстрагирует text-only branch loading,
    skipping vision tower (saves ~2GB VRAM + matches training stack exactly).

    Adapter merge_and_unload даёт plain HF model для inference path,
    FastLanguageModel.for_inference активирует Unsloth fast kernels.
    """
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=False,
        dtype=torch.bfloat16,
    )
    # Unsloth wraps tokenizer in some versions — unwrap if needed.
    if not hasattr(tokenizer, 'vocab_size') and hasattr(tokenizer, 'tokenizer'):
        tokenizer = tokenizer.tokenizer

    if adapter_id:
        model = PeftModel.from_pretrained(model, adapter_id)
        model = model.merge_and_unload()
        logger.info(f'Merged adapter {adapter_id}')

    FastLanguageModel.for_inference(model)
    return model, tokenizer


def generate_one(model, tokenizer, prompt: str) -> str:
    messages = [
        {'role': 'system', 'content': DECODING_CONFIG['system_prompt']},
        {'role': 'user', 'content': prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=DECODING_CONFIG['enable_thinking'],
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=DECODING_CONFIG['num_predict'],
            do_sample=DECODING_CONFIG['temperature'] > 0,
            temperature=max(DECODING_CONFIG['temperature'], 1e-5),
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    completion = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return completion


In [ ]:
# Cell 7: Eval loop (Phase 0a — fast_mode=True, programmatic correctness only)
# Phase 0b async judge — отдельным скриптом потом, когда Cerebras quota свободна.
import re

def _normalize_for_compare(s) -> str:
    """Normalize answer string for numeric/symbolic comparison.

    Accepts str | int | float | None. eval_dataset.jsonl держит numeric
    ground_truth как int/float (JSON не coerce-ит в строки), поэтому
    str(s) делается до strip().
    """
    if s is None:
        return ''
    s = str(s).strip()
    if not s:
        return ''
    if s.startswith('$') and s.endswith('$'):
        s = s[1:-1].strip()
    s = re.sub(r'\\text\s*\{[^}]*\}?', '', s)
    s = re.sub(r'\\mathrm\s*\{[^}]*\}?', '', s)
    # Russian decimal comma → dot, strip whitespace incl. nbsp
    s = s.replace(' ', '').replace(' ', '').replace(',', '.')
    # Strip trailing punctuation
    # If LaTeX equation residue (e.g. 'v=100' from $v=100	ext{м/с}$), take RHS
    if '=' in s:
        s = s.split('=')[-1].strip()
    s = s.rstrip('.;,')
    # FALLBACK: leading numeric token — handles broken/residual wrappers
    m = re.match(r'^[+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?', s)
    if m:
        return m.group(0)
    return s


def is_numeric_correct(extracted, ground_truth, tolerance: float = 0.02) -> bool | None:
    """Programmatic correctness check.

    - For numeric strings: relative tolerance 2% or absolute < 0.001 if truth ≈ 0.
    - For non-numeric: case-insensitive normalized string match.
    - Returns None if can't determine (empty extracted or both unparseable).
    """
    e_norm = _normalize_for_compare(extracted)
    g_norm = _normalize_for_compare(ground_truth)
    if not e_norm or not g_norm:
        return None
    try:
        ef, gf = float(e_norm), float(g_norm)
        if abs(gf) < 1e-9:
            return abs(ef) < 0.001
        return abs(ef - gf) / abs(gf) < tolerance
    except ValueError:
        return e_norm.lower() == g_norm.lower()


def eval_stage(stage_name: str, adapter_id: str | None, problems: List[Dict],
               fast_mode: bool = True) -> Dict[str, Any]:
    """Evaluate one stage.

    Args:
        fast_mode: Phase 0a (True) — programmatic correctness only, saves raw
                   completions для Phase 0b (async Cerebras judge).
                   Phase 0 original (False) — синхронный Cerebras judge per problem
                   (упирается в rate limits, не рекомендуется).
    """
    logger.info(f'=== Stage: {stage_name} ({adapter_id or "base"}) | fast_mode={fast_mode} ===')
    model, tokenizer = load_model_with_adapter(adapter_id)
    cerebras = None if fast_mode else CerebrasClient()
    completions = []
    t0 = time.time()
    for i, p in enumerate(problems):
        t_start = time.time()
        completion = generate_one(model, tokenizer, p['prompt'])
        t_gen = time.time() - t_start
        # Always compute — нужно для truncation/think_close tracking
        n_tokens = len(tokenizer.encode(completion))
        truncated = n_tokens >= DECODING_CONFIG['num_predict'] - 5
        has_think_close = '</think>' in completion
        # Per-problem timing для первых 3 — диагностика fast-path vs fallback скорости.
        if i < 3:
            logger.info(f'  [{i+1}/{len(problems)}] gen={t_gen:.1f}s | tokens={n_tokens} | tok/s={n_tokens/max(t_gen,0.01):.1f}')
        extracted = extract_answer(completion)
        fmt = check_format_compliance(completion)
        visible = completion.split('</think>')[-1].strip() if '</think>' in completion else completion

        if fast_mode:
            correct = is_numeric_correct(extracted, p['ground_truth'])
            socratic, no_leak = None, None
        else:
            judge = evaluate_combined_quality(p['prompt'], visible, p['ground_truth'], cerebras)
            correct = judge.get('is_correct')
            socratic = judge.get('socratic_score')
            no_leak = judge.get('no_answer_leak')

        completions.append({
            'idx': i, 'domain': p['domain'], 'difficulty': p['difficulty'],
            'truth': p['ground_truth'], 'extracted': extracted,
            'correct': correct,
            'socratic_score': socratic,
            'no_answer_leak': no_leak,
            'completion_text': visible,  # saved для Phase 0b async judge
            'n_tokens': n_tokens,
            'truncated': truncated,
            'has_think_close': has_think_close,
            **fmt,
        })

        log_every = 5 if fast_mode else 10
        if (i + 1) % log_every == 0:
            elapsed = time.time() - t0
            valid = [c for c in completions if c['correct'] is not None]
            acc = sum(1 for c in valid if c['correct']) / max(len(valid), 1)
            logger.info(f'  {i+1}/{len(problems)} | acc={acc:.3f} ({len(valid)} judged) | {elapsed/60:.1f} min')

    del model; torch.cuda.empty_cache()
    valid = [c for c in completions if c['correct'] is not None]
    accuracy = sum(1 for c in valid if c['correct']) / max(len(valid), 1)
    socratic_vals = [c['socratic_score'] for c in completions if c.get('socratic_score') is not None]
    leak_vals = [c['no_answer_leak'] for c in completions if c.get('no_answer_leak') is not None]
    return {
        'stage': stage_name, 'adapter': adapter_id, 'n': len(completions),
        'n_judged': len(valid),
        'accuracy': accuracy,
        'avg_socratic': (sum(socratic_vals) / len(socratic_vals)) if socratic_vals else None,
        'leak_rate': (sum(1 for v in leak_vals if v < 2) / len(leak_vals)) if leak_vals else None,
        'truncation_rate': sum(1 for c in completions if c.get('truncated')) / max(len(completions), 1),
        'think_close_rate': sum(1 for c in completions if c.get('has_think_close')) / max(len(completions), 1),
        'accuracy_non_truncated': (
            sum(1 for c in completions if c.get('correct') and not c.get('truncated'))
            / max(sum(1 for c in completions if c.get('correct') is not None and not c.get('truncated')), 1)
        ),
        'n_non_truncated': sum(1 for c in completions if not c.get('truncated')),
        'mode': 'fast_programmatic' if fast_mode else 'full_cerebras_judge',
        'completions': completions,
    }

In [ ]:
# Cell 7.5: Distribution diagnostic + optional subset selection.
# UPDATE: Empirical speed test showed steady-state 13.3 tok/s (Run 2/3 после
# Triton JIT compile ~70s одноразово). Full 143 × 3 stages feasible ~7.5h.
# Default: full eval. Uncomment LAST 2 lines чтобы переключиться на 30 stratified.
import random
from collections import defaultdict

random.seed(42)

by_diff = defaultdict(list)
for p in calc_problems:
    by_diff[p.get('difficulty', 'medium')].append(p)

logger.info(f'Calc problems by difficulty: {[(k, len(v)) for k, v in by_diff.items()]}')
logger.info(f'Default: full eval ({len(calc_problems)} problems × 3 stages ≈ 7.5h)')

# ── OPTIONAL: stratified 30 subset (если хочешь quick test ~2.5h) ──
# stratified = []
# for diff in ['easy', 'medium', 'hard']:
#     pool = by_diff.get(diff, [])
#     stratified.extend(random.sample(pool, min(10, len(pool))))
# calc_problems = stratified
# logger.info(f'OVERRIDE: using 30 stratified subset')


In [ ]:
# Cell SMOKE: 3-problem dry run — verify quality + measure realistic timing
# BEFORE committing к full 18h overnight eval.
# Picks one problem per difficulty, runs base model, prints completions для sanity check.
import time
import torch

# Load base model fresh (or reuse if already loaded в session)
try:
    _ = model.device
    logger.info(f'Reusing already-loaded model on {model.device}')
except (NameError, AttributeError):
    logger.info('Loading base model для smoke test...')
    model, tokenizer = load_model_with_adapter(None)

# Pick one problem per difficulty для diverse coverage
smoke_problems = []
seen_diffs = set()
for p in calc_problems:
    diff = p.get('difficulty', 'medium')
    if diff not in seen_diffs:
        smoke_problems.append(p)
        seen_diffs.add(diff)
    if len(smoke_problems) >= 3:
        break

logger.info(f'Smoke test: {len(smoke_problems)} problems (difficulties: {[p.get("difficulty") for p in smoke_problems]})')
logger.info(f'num_predict={DECODING_CONFIG["num_predict"]}, enable_thinking={DECODING_CONFIG["enable_thinking"]}')
print()

t_start = time.time()
smoke_results = []
for i, p in enumerate(smoke_problems):
    diff = p.get('difficulty', '?')
    print(f'─── Problem {i+1}/{len(smoke_problems)} [{diff}] ───')
    print(f'Q: {p["prompt"][:200]}{"..." if len(p["prompt"]) > 200 else ""}')
    print(f'Truth: {p["ground_truth"]}')

    t0 = time.time()
    completion = generate_one(model, tokenizer, p['prompt'])
    t_gen = time.time() - t0

    # Per-problem stats
    n_total = len(tokenizer.encode(completion))
    has_think_close = '</think>' in completion
    visible = completion.split('</think>')[-1].strip() if has_think_close else completion
    extracted = extract_answer(completion)
    correct = is_numeric_correct(extracted, p['ground_truth'])
    truncated = n_total >= DECODING_CONFIG['num_predict'] - 5  # within 5 tokens of cap

    print(f'Time: {t_gen:.1f}s | tokens: {n_total} | tok/s: {n_total/max(t_gen, 0.01):.1f}')
    print(f'Has </think>: {has_think_close} | Truncated: {truncated} | Extracted: {extracted!r} | Correct: {correct}')
    print(f'\n[Visible answer (first 400 chars)]:')
    print(visible[:400] + ('...' if len(visible) > 400 else ''))
    print()

    smoke_results.append({
        'idx': i, 'difficulty': diff, 'time_s': t_gen, 'tokens': n_total,
        'has_think_close': has_think_close, 'truncated': truncated,
        'extracted': extracted, 'truth': p['ground_truth'], 'correct': correct,
    })

t_total = time.time() - t_start
print('═' * 60)
print(f'Smoke test complete: {t_total:.1f}s for {len(smoke_problems)} problems')
print()

# Calculate full eval ETA
avg_time = sum(r['time_s'] for r in smoke_results) / len(smoke_results)
truncation_rate = sum(1 for r in smoke_results if r['truncated']) / len(smoke_results)
correct_count = sum(1 for r in smoke_results if r['correct'])

print(f'Per-problem avg: {avg_time:.1f}s')
print(f'Truncation rate: {truncation_rate:.1%}')
print(f'Quick accuracy: {correct_count}/{len(smoke_results)} (sample size мал, indicative only)')
print()
print(f'ETA for full eval (143 × 3 stages): {143 * avg_time * 3 / 3600:.1f}h overnight')
print()

# Sanity flags
flags = []
if avg_time > 200:
    flags.append('SLOW: avg >200s per problem — full eval >24h')
if truncation_rate > 0.3:
    flags.append('TRUNCATION: >30% hit cap — increase num_predict or accept loss')
if not all(r['has_think_close'] for r in smoke_results):
    flags.append("NO </think>: model не emit close tag — truncation или training issue")

if flags:
    print('⚠ FLAGS:')
    for f in flags:
        print(f'  - {f}')
    print('\nDecide: continue к full eval (Cell next) OR adjust DECODING_CONFIG and re-run smoke.')
else:
    print('✓ All checks pass. Proceed to full eval (next cell).')


In [ ]:
# Cell 9: Run Phase 0a with per-stage incremental save (resumable).
# Каждый stage saved immediately после completion → disconnect-resilient.
# Resume: re-run эту cell — completed stages loaded from disk, остальные re-run.
date_tag = datetime.utcnow().strftime('%Y%m%d')
stage_dir = PROJECT_ROOT / f'evaluation/reports/honest_phase0a_{date_tag}'
stage_dir.mkdir(parents=True, exist_ok=True)
drive_root = Path('/content/drive/MyDrive/MITS_secrets')
drive_stage_dir = drive_root / f'honest_phase0a_{date_tag}' if drive_root.exists() else None
if drive_stage_dir is not None:
    drive_stage_dir.mkdir(parents=True, exist_ok=True)

logger.info(f'Per-stage save dir (repo): {stage_dir}')
if drive_stage_dir:
    logger.info(f'Per-stage save dir (Drive): {drive_stage_dir}')

results = {}
for stage_name, adapter_id in ADAPTERS.items():
    stage_file = stage_dir / f'{stage_name}.json'
    drive_stage_file = (drive_stage_dir / f'{stage_name}.json') if drive_stage_dir else None

    # Resume: skip if already saved on disk (или Drive — restore оттуда)
    if stage_file.exists():
        logger.info(f'[RESUME] {stage_name} already saved at {stage_file}, loading')
        results[stage_name] = json.loads(stage_file.read_text(encoding='utf-8'))
        continue
    if drive_stage_file is not None and drive_stage_file.exists():
        logger.info(f'[RESUME from Drive] {stage_name} loading from {drive_stage_file}')
        results[stage_name] = json.loads(drive_stage_file.read_text(encoding='utf-8'))
        # Mirror back к repo for git-tracked artifact
        stage_file.write_text(drive_stage_file.read_text(encoding='utf-8'), encoding='utf-8')
        continue

    logger.info(f'>>> Starting stage: {stage_name}')
    r = eval_stage(stage_name, adapter_id, calc_problems, fast_mode=True)
    results[stage_name] = r

    # Save IMMEDIATELY — repo + Drive
    stage_file.write_text(json.dumps(r, ensure_ascii=False, indent=2), encoding='utf-8')
    logger.info(f'[SAVED] {stage_file} ({stage_file.stat().st_size / 1024:.1f} KB)')
    if drive_stage_file is not None:
        drive_stage_file.write_text(json.dumps(r, ensure_ascii=False, indent=2), encoding='utf-8')
        logger.info(f'[SAVED Drive] {drive_stage_file}')

# All stages complete — assemble final merged report
report = {
    'protocol': 'honest_full_precision_phase0a',
    'mode': 'fast_programmatic',
    'note': (
        'Phase 0a: accuracy via programmatic numeric/string match (extract_answer vs '
        'ground_truth, 2% tolerance). Per-stage incremental save в stage_dir. '
        'socratic_score / leak_rate deferred to Phase 0b (async Cerebras judge).'
    ),
    'timestamp': datetime.utcnow().isoformat(),
    'decoding_config': {k: v for k, v in DECODING_CONFIG.items() if k != 'system_prompt'},
    'system_prompt_hash': hash(DECODING_CONFIG['system_prompt']),
    'n_problems': len(calc_problems),
    'base': {k: v for k, v in results['base'].items() if k != 'completions'},
    'gspo': {k: v for k, v in results['gspo'].items() if k != 'completions'},
    'kto':  {k: v for k, v in results['kto'].items()  if k != 'completions'},
    'completions': {stage: r['completions'] for stage, r in results.items()},
}

out_repo  = PROJECT_ROOT / f'evaluation/reports/honest_full_precision_phase0a_{date_tag}.json'
out_drive = drive_root / f'honest_full_precision_phase0a_{date_tag}.json' if drive_root.exists() else None
out_repo.parent.mkdir(parents=True, exist_ok=True)
out_repo.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
logger.info(f'Saved merged report (repo): {out_repo}')
if out_drive:
    out_drive.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    logger.info(f'Saved merged report (Drive): {out_drive}')

print('\n=== Phase 0a — Honest accuracy (full-precision bf16, identical decoding, programmatic correctness) ===')
for stage in ['base', 'gspo', 'kto']:
    r = results[stage]
    print(f"{stage:5s}  acc={r['accuracy']:.3f} ({r['n_judged']}/{r['n']} judged) | "
          f"acc_non_trunc={r['accuracy_non_truncated']:.3f} ({r['n_non_truncated']}/{r['n']}) | "
          f"trunc={r['truncation_rate']:.1%} | think_close={r['think_close_rate']:.1%}")
print('\nNote: acc_non_trunc — accuracy ON completed answers только (excluded truncated).')
print('Note: socratic_score / leak_rate — Phase 0b (run scripts/score_phase0b_async.py later).')
